# M2 emotion training on Colab

Runtime > Change runtime type > T4 GPU. Runs the three M2 configs and saves
runs + reports to Google Drive so nothing is lost when the session ends.

FER2013 data: put one of these in `MyDrive/interviewlens/` (recommended, it
survives the session) or upload it when the data cell asks:
`fer2013.csv`, `icml_face_data.csv`, `fer2013.tar.gz`, or the full competition
zip from
https://www.kaggle.com/c/challenges-in-representation-learning-facial-expression-recognition-challenge/data

It has to be the competition csv version (pixels + Usage columns). The
image-folder "FER2013" datasets on Kaggle don't work: they lack the
PublicTest/PrivateTest split and the row order FER+ labels rely on.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/interviewlens"
!mkdir -p {DRIVE}

In [ ]:
!git clone -q https://github.com/mohanmarimuthu1/Interview_Analyzer.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -e ".[train]"

## Data

Looks in Drive first, otherwise asks for an upload. The extracted csv is
cached on Drive as `fer2013.csv` for later sessions.

In [ ]:
import glob, os, shutil, tarfile, zipfile

RAW = "data/raw/fer2013"
os.makedirs(RAW, exist_ok=True)
NAMES = (
    "fer2013.csv",
    "icml_face_data.csv",
    "fer2013.tar.gz",
    "challenges-in-representation-learning-facial-expression-recognition-challenge.zip",
)

src = next((f"{DRIVE}/{n}" for n in NAMES if os.path.exists(f"{DRIVE}/{n}")), None)
if src is None:
    from google.colab import files
    print(f"nothing in {DRIVE}, pick the file to upload")
    src = os.path.abspath(next(iter(files.upload())))
print("using", src)

if src.endswith(".zip"):
    zipfile.ZipFile(src).extractall(RAW)
elif src.endswith(".tar.gz"):
    tarfile.open(src).extractall(RAW)
else:
    shutil.copy(src, RAW)
for tgz in glob.glob(f"{RAW}/**/fer2013.tar.gz", recursive=True):  # inside the full zip
    tarfile.open(tgz).extractall(RAW)

csvs = [c for n in ("fer2013.csv", "icml_face_data.csv") for c in glob.glob(f"{RAW}/**/{n}", recursive=True)]
if not csvs:
    raise FileNotFoundError("no fer2013.csv / icml_face_data.csv found - is this the competition csv version?")
if os.path.abspath(csvs[0]) != os.path.abspath(f"{RAW}/fer2013.csv"):
    shutil.copy(csvs[0], f"{RAW}/fer2013.csv")
if not os.path.exists(f"{DRIVE}/fer2013.csv"):
    shutil.copy(f"{RAW}/fer2013.csv", f"{DRIVE}/fer2013.csv")

!mkdir -p data/raw/ferplus
!curl -sfL -o data/raw/ferplus/fer2013new.csv https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv
!wc -l data/raw/fer2013/fer2013.csv data/raw/ferplus/fer2013new.csv

## Runs

1. ResNet-18 on FER2013 labels: v1 parity check (target >= 68% test acc)
2. ResNet-18 on FER+ labels
3. ConvNeXt-T on FER+ labels (lower lr, it is a bigger pretrained model)

Each run: best epoch picked on PublicTest, reported on PrivateTest.

In [ ]:
RUN = f"hydra.run.dir={DRIVE}/runs/" + "${run_name}"
REPORTS = f"reports_dir={DRIVE}/reports/emotion"
!python -m interviewlens.train_emotion data.labels=fer2013 model=resnet18 '{RUN}' {REPORTS}

In [ ]:
!python -m interviewlens.train_emotion data.labels=ferplus model=resnet18 '{RUN}' {REPORTS}

In [ ]:
!python -m interviewlens.train_emotion data.labels=ferplus model=convnext_t train.lr=1e-4 train.batch_size=64 '{RUN}' {REPORTS}

## Results

Download `reports/emotion/` from Drive into the repo's `reports/emotion/`
(metrics.json + confusion.png per run). Checkpoints stay in `runs/` on Drive.

In [ ]:
import glob, json
for p in sorted(glob.glob(f"{DRIVE}/reports/emotion/*/metrics.json")):
    m = json.load(open(p))
    print(f"{m['run']:32s} val {m['best_val_acc']:.4f}  test acc {m['test_acc']:.4f}  macro-F1 {m['test_macro_f1']:.4f}")